Experiment 7: Sequence-to-Sequence Model with Real Text Data (07_seq2seq)
Aim

To implement a Sequence-to-Sequence (Seq2Seq) model using LSTM with real text sequence data in TensorFlow and Keras.



In [1]:
import tensorflow as tf
import numpy as np

from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input
from tensorflow.keras.layers import LSTM
from tensorflow.keras.layers import Dense
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Sample Text Dataset
input_texts = [
    "hello",
    "how are you",
    "good morning",
    "thank you",
    "see you"
]

target_texts = [
    "hi",
    "i am fine",
    "morning",
    "welcome",
    "bye"
]

# Tokenizer
tokenizer = Tokenizer()

tokenizer.fit_on_texts(input_texts + target_texts)

# Convert Text to Sequences
input_sequences = tokenizer.texts_to_sequences(input_texts)
target_sequences = tokenizer.texts_to_sequences(target_texts)

# Padding
max_len = max(
    max(len(seq) for seq in input_sequences),
    max(len(seq) for seq in target_sequences)
)

encoder_input_data = pad_sequences(
    input_sequences,
    maxlen=max_len,
    padding='post'
)

decoder_input_data = pad_sequences(
    target_sequences,
    maxlen=max_len,
    padding='post'
)

# Vocabulary Size
vocab_size = len(tokenizer.word_index) + 1

# One Hot Encode Target
decoder_target_data = tf.keras.utils.to_categorical(
    decoder_input_data,
    num_classes=vocab_size
)

# Parameters
latent_dim = 64

# Encoder
encoder_inputs = Input(shape=(max_len,))

encoder_embedding = tf.keras.layers.Embedding(
    vocab_size,
    latent_dim
)(encoder_inputs)

encoder_lstm = LSTM(
    latent_dim,
    return_state=True
)

encoder_outputs, state_h, state_c = encoder_lstm(
    encoder_embedding
)

encoder_states = [state_h, state_c]

# Decoder
decoder_inputs = Input(shape=(max_len,))

decoder_embedding = tf.keras.layers.Embedding(
    vocab_size,
    latent_dim
)(decoder_inputs)

decoder_lstm = LSTM(
    latent_dim,
    return_sequences=True,
    return_state=True
)

decoder_outputs, _, _ = decoder_lstm(
    decoder_embedding,
    initial_state=encoder_states
)

decoder_dense = Dense(
    vocab_size,
    activation='softmax'
)

decoder_outputs = decoder_dense(decoder_outputs)

# Seq2Seq Model
model = Model(
    [encoder_inputs, decoder_inputs],
    decoder_outputs
)

# Model Summary
model.summary()

# Compile Model
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Train Model
history = model.fit(
    [encoder_input_data, decoder_input_data],
    decoder_target_data,
    batch_size=2,
    epochs=200,
    verbose=1
)

# Evaluate Model
loss, accuracy = model.evaluate(
    [encoder_input_data, decoder_input_data],
    decoder_target_data
)

print("\nModel Loss :", loss)
print("Model Accuracy :", accuracy)

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 3)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_1       │ (None, 3)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 3, 64)     │        960 │ input_layer[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_1         │ (None, 3, 64)     │        960 │ input_layer_1[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm (LSTM)         │ [(None, 64),      │     33,024 │ embedding[0][0]   │
│                     │ (None, 64),       │            │                   │
│                     │ (None, 64)]       │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_1 (LSTM)       │ [(None, 3, 64),   │     33,024 │ embedding_1[0][0… │
│                     │ (None, 64),       │            │ lstm[0][1],       │
│                     │ (None, 64)]       │            │ lstm[0][2]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 3, 15)     │        975 │ lstm_1[0][0]      │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 68,943 (269.31 KB)

 Trainable params: 68,943 (269.31 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 4s 19ms/step - accuracy: 0.3333 - loss: 2.7004  
Epoch 2/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.5333 - loss: 2.6670
Epoch 3/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.5333 - loss: 2.6277
Epoch 4/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.5333 - loss: 2.5871
Epoch 5/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.5333 - loss: 2.5426
Epoch 6/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - accuracy: 0.5333 - loss: 2.4742
Epoch 7/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.5333 - loss: 2.3909
Epoch 8/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.5333 - loss: 2.2773
Epoch 9/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.5333 - loss: 2.1104
Epoch 10/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.5333 - loss: 1.9031
Epoch 11/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.5333 - loss: 1.6807
Epoch 12/200
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.5333 - 

Conclusion

Successfully implemented a Sequence-to-Sequence (Seq2Seq) model using real text sequence data with Encoder-Decoder LSTM architecture in TensorFlow and Keras.